# 01 — Exploratory analysis of synthetic restructuring outcomes

This notebook contains **no historical observations**. The dataset is generated by the simulator and is not a forecast of real restructuring. Run the README quick-start commands first.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

path = Path("output/demo/empresas_resultados.csv")
if not path.exists():
    raise FileNotFoundError("Run the synthetic demo commands from the repository root first")
df = pd.read_csv(path)
assert not df.empty and df["company_id"].astype(str).str.startswith("SYNT-").all()
print("Synthetic records:", len(df))
df.head()

## Data dictionary and consistency checks

We distinguish **control transfer** from **operational autonomy**. A simulated success flag is not legal compliance.

In [ ]:
cols = ["category", "control_transfer_month", "operational_autonomy_month", "autonomy_by_maximum", "legal_compliance_established"]
assert (df["operational_autonomy_month"] >= df["control_transfer_month"]).all()
assert (df["legal_compliance_established"] == 0).all()
df[cols].describe(include="all")

## Compare durations by synthetic complexity category

In [ ]:
df.groupby("category")["operational_autonomy_month"].agg(["count", "median", "mean", "min", "max"]).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9,4.5))
for category, group in sorted(df.groupby("category")):
    ax.hist(group["operational_autonomy_month"], bins=16, alpha=0.55, label=f"Category {category} (n={len(group)})")
ax.set(xlabel="Simulated months", ylabel="Synthetic companies", title="Distribution of operational autonomy")
ax.legend()
plt.show()

## Events and diagnostic correlations

The incident flags are generated by the model; their rates are not historical incident rates.

In [ ]:
flags = [c for c in df if c.endswith("_hypothetical")]
df[flags].mean().sort_values(ascending=False).to_frame("synthetic_incident_fraction")

In [ ]:
features=["complexity_technical", "complexity_dependencies", "complexity_labor", "complexity_service_continuity", "operational_autonomy_month"]
df[features].corr(numeric_only=True).round(2)

## Interpretation

These statistics explore how the implemented **assumptions** behave. They cannot establish independent forecasting skill, policy effectiveness, or legal compliance. Compare configuration changes in the second notebook.